# 03 - Prompt Versioning

Prompt versioning system, A/B testing, performance tracking, regression testing, and prompt registry.


In [ ]:
import numpy as np
import re
import json
import time
import matplotlib.pyplot as plt
from collections import defaultdict
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded!')


## 1. Prompt Versioning System


In [ ]:
class PromptVersion:
    def __init__(self, prompt_id, content, version, author, tags=None):
        self.prompt_id = prompt_id
        self.content = content
        self.version = version
        self.author = author
        self.created_at = datetime.now().isoformat()
        self.tags = tags or []
        self.metrics = {}
    def to_dict(self):
        return {
            'prompt_id': self.prompt_id,
            'version': self.version,
            'content': self.content,
            'author': self.author,
            'created_at': self.created_at,
            'tags': self.tags,
            'metrics': self.metrics
        }

class PromptRegistry:
    def __init__(self):
        self.prompts = defaultdict(list)
    def register(self, prompt_version):
        self.prompts[prompt_version.prompt_id].append(prompt_version)
        print(f'Registered {prompt_version.prompt_id} v{prompt_version.version}')
    def get_version(self, prompt_id, version):
        for pv in self.prompts[prompt_id]:
            if pv.version == version:
                return pv
        return None
    def get_latest(self, prompt_id):
        if prompt_id in self.prompts and self.prompts[prompt_id]:
            return sorted(self.prompts[prompt_id], key=lambda x: x.version)[-1]
        return None
    def list_versions(self, prompt_id):
        return [pv.version for pv in self.prompts.get(prompt_id, [])]

registry = PromptRegistry()

v1 = PromptVersion('sentiment_classifier', 'Classify sentiment: {text}', 1.0, 'alice', ['nlp', 'classification'])
v2 = PromptVersion('sentiment_classifier', 'Classify the sentiment of the following text as Positive, Negative, or Neutral:\n{text}', 2.0, 'bob', ['nlp', 'classification', 'improved'])
v3 = PromptVersion('sentiment_classifier', 'Analyze the emotional tone of this text. Respond with exactly one word: Positive, Negative, or Neutral.\n\nText: {text}\nTone:', 3.0, 'charlie', ['nlp', 'classification', 'structured'])

registry.register(v1)
registry.register(v2)
registry.register(v3)

print(f'\nVersions: {registry.list_versions("sentiment_classifier")}')


## 2. A/B Testing Framework


In [ ]:
class ABTest:
    def __init__(self, test_name, prompt_a, prompt_b):
        self.test_name = test_name
        self.prompt_a = prompt_a
        self.prompt_b = prompt_b
        self.results_a = []
        self.results_b = []
        self.assignments = []
    def assign_variant(self, user_id):
        variant = 'A' if hash(user_id) % 2 == 0 else 'B'
        self.assignments.append({'user_id': user_id, 'variant': variant})
        return variant
    def record_result(self, variant, accuracy, latency, user_satisfaction):
        result = {'accuracy': accuracy, 'latency': latency, 'satisfaction': user_satisfaction}
        if variant == 'A':
            self.results_a.append(result)
        else:
            self.results_b.append(result)
    def get_stats(self):
        def avg(key, results):
            return np.mean([r[key] for r in results]) if results else 0
        return {
            'A': {'count': len(self.results_a), 'accuracy': avg('accuracy', self.results_a), 'latency': avg('latency', self.results_a), 'satisfaction': avg('satisfaction', self.results_a)},
            'B': {'count': len(self.results_b), 'accuracy': avg('accuracy', self.results_b), 'latency': avg('latency', self.results_b), 'satisfaction': avg('satisfaction', self.results_b)}
        }
    def winner(self):
        stats = self.get_stats()
        score_a = stats['A']['accuracy'] * 0.5 + stats['A']['satisfaction'] * 0.5
        score_b = stats['B']['accuracy'] * 0.5 + stats['B']['satisfaction'] * 0.5
        return 'A' if score_a > score_b else 'B'

ab = ABTest('sentiment_v2_vs_v3', v2.content, v3.content)

np.random.seed(42)
for i in range(100):
    user_id = f'user_{i}'
    variant = ab.assign_variant(user_id)
    if variant == 'A':
        ab.record_result('A', np.random.normal(0.78, 0.05), np.random.normal(0.3, 0.1), np.random.normal(3.8, 0.3))
    else:
        ab.record_result('B', np.random.normal(0.85, 0.04), np.random.normal(0.25, 0.08), np.random.normal(4.2, 0.2))

stats = ab.get_stats()
print(f'A/B Test Results for {ab.test_name}:')
for variant, data in stats.items():
    print(f'  Variant {variant}: {data}')
print(f'\nWinner: Variant {ab.winner()}')


## 3. Performance Tracking


In [ ]:
class PerformanceTracker:
    def __init__(self):
        self.history = defaultdict(list)
    def record(self, prompt_id, version, metrics):
        self.history[(prompt_id, version)].append({
            'timestamp': datetime.now().isoformat(),
            **metrics
        })
    def get_trend(self, prompt_id, version, metric='accuracy'):
        key = (prompt_id, version)
        if key not in self.history:
            return []
        return [h[metric] for h in self.history[key]]
    def plot_trend(self, prompt_id, versions, metric='accuracy'):
        plt.figure(figsize=(10, 5))
        for v in versions:
            trend = self.get_trend(prompt_id, v, metric)
            if trend:
                plt.plot(trend, marker='o', label=f'v{v}')
        plt.xlabel('Evaluation Run')
        plt.ylabel(metric.title())
        plt.title(f'Performance Trend: {prompt_id}')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

tracker = PerformanceTracker()

for run in range(10):
    tracker.record('sentiment_classifier', 2.0, {'accuracy': 0.75 + run * 0.01 + np.random.normal(0, 0.02), 'latency': 0.3 - run * 0.01})
    tracker.record('sentiment_classifier', 3.0, {'accuracy': 0.80 + run * 0.015 + np.random.normal(0, 0.015), 'latency': 0.25 - run * 0.008})

tracker.plot_trend('sentiment_classifier', [2.0, 3.0], 'accuracy')


## 4. Regression Testing


In [ ]:
class RegressionTest:
    def __init__(self, test_cases):
        self.test_cases = test_cases
        self.results = []
    def run(self, prompt_fn):
        passed = 0
        for inp, expected in self.test_cases:
            output = prompt_fn(inp)
            match = expected.lower() in output.lower()
            self.results.append({'input': inp, 'expected': expected, 'output': output, 'passed': match})
            if match:
                passed += 1
        return passed / len(self.test_cases)

test_cases = [
    ('I love this!', 'positive'),
    ('This is terrible.', 'negative'),
    ('It is okay.', 'neutral'),
    ('Amazing product!', 'positive'),
    ('Not worth the money.', 'negative')
]

def mock_sentiment_prompt(text):
    positive_words = ['love', 'amazing', 'great', 'excellent', 'good']
    negative_words = ['terrible', 'bad', 'hate', 'worst', 'not worth']
    text_lower = text.lower()
    if any(w in text_lower for w in positive_words):
        return 'positive'
    if any(w in text_lower for w in negative_words):
        return 'negative'
    return 'neutral'

reg_test = RegressionTest(test_cases)
pass_rate = reg_test.run(mock_sentiment_prompt)
print(f'Regression Test Pass Rate: {pass_rate*100:.1f}%')
print('\nDetailed results:')
for r in reg_test.results:
    status = 'PASS' if r['passed'] else 'FAIL'
    print(f'  [{status}] Input: {r["input"]} -> Expected: {r["expected"]}, Got: {r["output"]}')


## 5. Prompt Registry Dashboard


In [ ]:
def visualize_registry(registry):
    prompt_ids = list(registry.prompts.keys())
    version_counts = [len(registry.prompts[pid]) for pid in prompt_ids]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].bar(prompt_ids, version_counts, color='steelblue')
    axes[0].set_ylabel('Number of Versions')
    axes[0].set_title('Prompts in Registry')
    axes[0].tick_params(axis='x', rotation=15)
    variants = ['A', 'B']
    accuracies = [stats['A']['accuracy'], stats['B']['accuracy']]
    satisfactions = [stats['A']['satisfaction'], stats['B']['satisfaction']]
    x = np.arange(len(variants))
    width = 0.35
    axes[1].bar(x - width/2, accuracies, width, label='Accuracy', color='coral')
    axes[1].bar(x + width/2, satisfactions, width, label='Satisfaction', color='lightgreen')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(variants)
    axes[1].set_ylabel('Score')
    axes[1].set_title('A/B Test Comparison')
    axes[1].legend()
    plt.tight_layout()
    plt.show()

visualize_registry(registry)


## 6. Exercises


In [ ]:
# Exercise 1: Add rollback functionality
# Exercise 2: Implement prompt diff viewer
# Exercise 3: Add automated regression alerts
# Exercise 4: Build a prompt approval workflow
print('Exercises listed!')
